### Install Packages

In [ ]:
!pip install tabpfn openml contrastive tqdm

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 173.7/173.7 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.4/160.4 kB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.1/93.1 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.3/137.3 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 105.0 MB/s eta 0:00:00
  Created wheel for contrastive: filename=contrastive-1.2.0-py3-none-any.whl size=6899 sha256=7cbff132665aba513cf5b288c02dc8fec23256ba3ea4450f340a885a01cb64ec
  Stored in directory: /root/.cache/pip/wheels/c0/b5/06/f9cee153dd8a0cd9f13a325b1f8cae67bf546c2d83ca98d736
  Created wheel for liac-arff: filename=liac_arff-2.5.0-py3-none-any.whl size=11717 sha256=7e3e648e7d8c971dca41f1821da9b11359d8b4c25cdf6150aea2099c513eb7bc
  Stored in direct

In [ ]:
pip install git+https://github.com/soda-inria/tabicl.git

  Cloning https://github.com/soda-inria/tabicl.git to /tmp/pip-req-build-cfmeherp
  Running command git clone --filter=blob:none --quiet https://github.com/soda-inria/tabicl.git /tmp/pip-req-build-cfmeherp
  Resolved https://github.com/soda-inria/tabicl.git to commit c877e9155f6eb7bb562bef3a113a220c2badf22c
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for tabicl: filename=tabicl-0.1.3-py3-none-any.whl size=103918 sha256=603517ad668f5cf24f03e24b23546d761e49088b6130e0e922660dc844ecf9bb
  Stored in directory: /tmp/pip-ephem-wheel-cache-h10chrsu/wheels/3a/4a/56/e29394daa154d8d57213c99f51fdca7487f6d2209ab601b2eb
Successfully built tabicl


# Diabetes

In [1]:
import time
import numpy as np
import pandas as pd
from scipy.stats import sem
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, f1_score
from sklearn.impute import SimpleImputer
from contrastive import CPCA
from xgboost import XGBClassifier
from sklearn.svm import SVC
from tabpfn import TabPFNClassifier
from tabicl import TabICLClassifier


# ============ 0. SAVE RESULTS TO GOOGLE DRIVE ============
from google.colab import drive
drive.mount('/content/drive')
save_path = "/content/drive/MyDrive/diabetes_model_comparison_results.csv"

# ============ 1. LOAD DATASET ============
df = pd.read_csv("Dataset of Diabetes .csv")

# ============ 2. CLEAN & ENCODE ============
df = df.drop(columns=["ID", "No_Pation"], errors="ignore")
df["Gender"] = df["Gender"].map({"F": 0, "M": 1})

class_map = {"N": 0, "P": 1, "Y": 2}
df["CLASS_NUM"] = df["CLASS"].map(class_map)
df = df.dropna(subset=["CLASS_NUM"])

# Features and labels
X = df.drop(columns=["CLASS", "CLASS_NUM"])
y = df["CLASS_NUM"].values

# Impute missing values
imputer = SimpleImputer(strategy="mean")
X = imputer.fit_transform(X)

# ============ 3. SELECT FOREGROUND (P,Y) vs BACKGROUND (N) ============
mask_fg = y != 0  # P or Y
X_foreground = X[mask_fg]
y_foreground = y[mask_fg]
X_background = X[~mask_fg]
y_background = y[~mask_fg]

# ============ 4. TRAIN/TEST SPLIT ============
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X_foreground, y_foreground, test_size=0.2, random_state=42, stratify=y_foreground
)

# Fix labels if {1,2} → {0,1}
if sorted(np.unique(y_train_full)) == [1, 2]:
    y_train_full = (y_train_full - 1).astype(int)
    y_test = (y_test - 1).astype(int)

# ============ 5. STANDARDIZE ============
scaler = StandardScaler()
X_train_full = scaler.fit_transform(X_train_full)
X_test = scaler.transform(X_test)

# ============ 6. DEFINE MODELS ============
models = {
    "TabPFN": lambda: TabPFNClassifier(device="cpu", ignore_pretraining_limits=True),
    "TabICL": lambda: TabICLClassifier(),
    "XGBoost": lambda: XGBClassifier(
        n_estimators=300, max_depth=5, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, eval_metric="logloss", random_state=42
    ),
    "SVC": lambda: SVC(kernel="rbf", probability=False, random_state=42)
}

# ============ 7. SETTINGS ============
pca_components = [2, 10]
cpca_alphas = [1, 10, 100]
n_splits = 5
random_state = 42

# ============ 8. CROSS-VALIDATION ============
results = []
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)

for model_name, model_fn in models.items():
    print(f"\n=== Evaluating {model_name} ===")

    feature_sets = {"raw": X_train_full}

    # --- PCA variants ---
    for n_comp in pca_components:
        pca = PCA(n_components=n_comp, random_state=42)
        feature_sets[f"pca_{n_comp}"] = pca.fit_transform(X_train_full)

    # --- cPCA variants (vary n_components and alpha) ---
    bg_mask = np.zeros_like(y_train_full, dtype=bool)
    bg_mask[:len(y_background)] = True
    fg_mask = ~bg_mask

    # background: healthy (class 0), so use background data from earlier
    cpca = CPCA(n_components=max(pca_components), standardize=True)
    cpca.fit(X_foreground, X_background)

    for n_comp in pca_components:
        for alpha in cpca_alphas:
            transformed_list, _ = cpca.transform(X_train_full, alpha_value=alpha, return_alphas=True)
            X_cpca = transformed_list[0][:, :n_comp]
            feature_sets[f"cpca_{n_comp}_alpha{alpha}"] = X_cpca

    # --- Run CV ---
    for feat_name, X_data in feature_sets.items():
        print(f"\n>> {model_name} on {feat_name}")
        accs, f1s, times = [], [], []

        for train_idx, val_idx in skf.split(X_data, y_train_full):
            X_train, X_val = X_data[train_idx], X_data[val_idx]
            y_train, y_val = y_train_full[train_idx], y_train_full[val_idx]

            model = model_fn()
            start = time.time()
            model.fit(X_train, y_train)
            y_pred = model.predict(X_val)
            elapsed = time.time() - start

            accs.append(accuracy_score(y_val, y_pred))
            f1s.append(f1_score(y_val, y_pred, average="weighted"))
            times.append(elapsed)

        results.append({
            "Model": model_name,
            "Features": feat_name,
            "Mean_Acc": np.mean(accs),
            "Std_Acc": np.std(accs),
            "SE_Acc": sem(accs),
            "Mean_F1": np.mean(f1s),
            "Std_F1": np.std(f1s),
            "SE_F1": sem(f1s),
            "Mean_Time(s)": np.mean(times)
        })

# ============ 9. REFORMAT RESULTS ============
results_df = pd.DataFrame(results)

def parse_features(name):
    if name == "raw":
        return "None", 0, "N/A"
    elif name.startswith("pca_"):
        n = int(name.split("_")[1])
        return "PCA", n, "N/A"
    elif name.startswith("cpca_"):
        parts = name.split("_")
        n = int(parts[1])
        alpha = parts[2].replace("alpha", "")
        return "cPCA", n, alpha
    else:
        return "Unknown", "N/A", "N/A"

results_df[["preprocessing", "n_components", "alpha"]] = results_df["Features"].apply(
    lambda x: pd.Series(parse_features(x))
)

results_df.rename(columns={
    "Model": "model",
    "Mean_Acc": "accuracy",
    "Mean_F1": "f1_score",
    "Std_Acc": "std_accuracy",
    "Std_F1": "std_f1",
    "SE_Acc": "se_accuracy",
    "SE_F1": "se_f1",
    "Mean_Time(s)": "time_elapsed"
}, inplace=True)

results_df = results_df[[
    "model", "preprocessing", "n_components", "alpha",
    "accuracy", "f1_score",
    "std_accuracy", "std_f1",
    "se_accuracy", "se_f1", "time_elapsed"
]]

results_df = results_df.round(3)

# ============ 10. SAVE TO GOOGLE DRIVE ============
pd.set_option("display.max_rows", None)
print("\n=== Final Table (Publication Format) ===")
print(results_df)

results_df.to_csv(save_path, index=False)
print(f"\n✅ Results saved to Google Drive at: {save_path}")

ModuleNotFoundError: No module named 'contrastive'

# Gene expression

In [ ]:
import time
import numpy as np
import pandas as pd
from scipy.stats import sem
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, f1_score
from sklearn.impute import SimpleImputer
from contrastive import CPCA
from xgboost import XGBClassifier
from sklearn.svm import SVC
from tabpfn import TabPFNClassifier
from tabicl import TabICLClassifier

# ============ 1. LOAD DATA ============
df = pd.read_csv("gene_expression_for_cpca.csv", header=None)

# ============ 2. HANDLE LABELS & FEATURES ============
label_col = df.columns[-1]
df = df.dropna(subset=[label_col])

X = df.iloc[:, :-1]
y = df.iloc[:, -1].astype(int)

# Impute missing values
imputer = SimpleImputer(strategy="mean")
X_imputed = imputer.fit_transform(X)

# ============ 3. FOREGROUND (1,2) VS BACKGROUND (0) ============
mask_fg = np.isin(y, [1, 2])
X_foreground = X_imputed[mask_fg]
y_foreground = y.to_numpy()[mask_fg]
X_background = X_imputed[~mask_fg]
y_background = y.to_numpy()[~mask_fg]

# ============ 4. TRAIN/TEST SPLIT ============
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X_foreground, y_foreground, test_size=0.2, random_state=42, stratify=y_foreground
)

# Fix labels if {1,2} → {0,1}
if sorted(np.unique(y_train_full)) == [1, 2]:
    y_train_full = (y_train_full - 1).astype(int)
    y_test = (y_test - 1).astype(int)

# ============ 5. STANDARDIZE ============
scaler = StandardScaler()
X_train_full = scaler.fit_transform(X_train_full)
X_test = scaler.transform(X_test)

# ============ 6. DEFINE MODELS ============
models = {
    "TabPFN": lambda: TabPFNClassifier(device="cpu", ignore_pretraining_limits=True),
    "TabICL": lambda: TabICLClassifier(),
    "XGBoost": lambda: XGBClassifier(
        n_estimators=300, max_depth=5, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, eval_metric="logloss", random_state=42
    ),
    "SVC": lambda: SVC(kernel="rbf", probability=False, random_state=42)
}

# ============ 7. SETTINGS ============
pca_components = [2, 10, 20, 50]
cpca_alphas = [1, 10, 100]
n_splits = 5
random_state = 42

# ============ 8. CROSS-VALIDATION ============
results = []
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)

for model_name, model_fn in models.items():
    print(f"\n=== Evaluating {model_name} ===")

    feature_sets = {"raw": X_train_full}

    # --- PCA variants ---
    for n_comp in pca_components:
        pca = PCA(n_components=n_comp, random_state=42)
        feature_sets[f"pca_{n_comp}"] = pca.fit_transform(X_train_full)

    # --- cPCA variants: vary n_components and alpha ---
    cpca = CPCA(n_components=max(pca_components), standardize=True)
    cpca.fit(X_foreground, X_background)

    for n_comp in pca_components:
        for alpha in cpca_alphas:
            transformed_list, _ = cpca.transform(X_train_full, alpha_value=alpha, return_alphas=True)
            X_cpca = transformed_list[0][:, :n_comp]
            feature_sets[f"cpca_{n_comp}_alpha{alpha}"] = X_cpca

    # --- Run CV ---
    for feat_name, X_data in feature_sets.items():
        print(f"\n>> {model_name} on {feat_name}")
        accs, f1s, times = [], [], []

        for train_idx, val_idx in skf.split(X_data, y_train_full):
            X_train, X_val = X_data[train_idx], X_data[val_idx]
            y_train, y_val = y_train_full[train_idx], y_train_full[val_idx]

            model = model_fn()
            start = time.time()
            model.fit(X_train, y_train)
            y_pred = model.predict(X_val)
            elapsed = time.time() - start

            accs.append(accuracy_score(y_val, y_pred))
            f1s.append(f1_score(y_val, y_pred, average="weighted"))
            times.append(elapsed)

        results.append({
            "Model": model_name,
            "Features": feat_name,
            "Mean_Acc": np.mean(accs),
            "Std_Acc": np.std(accs),
            "SE_Acc": sem(accs),
            "Mean_F1": np.mean(f1s),
            "Std_F1": np.std(f1s),
            "SE_F1": sem(f1s),
            "Mean_Time(s)": np.mean(times)
        })

# ============ 9. REFORMAT RESULTS ============
results_df = pd.DataFrame(results)

def parse_features(name):
    if name == "raw":
        return "None", 0, "N/A"
    elif name.startswith("pca_"):
        n = int(name.split("_")[1])
        return "PCA", n, "N/A"
    elif name.startswith("cpca_"):
        parts = name.split("_")
        n = int(parts[1])
        alpha = parts[2].replace("alpha", "")
        return "cPCA", n, alpha
    else:
        return "Unknown", "N/A", "N/A"

results_df[["preprocessing", "n_components", "alpha"]] = results_df["Features"].apply(
    lambda x: pd.Series(parse_features(x))
)

results_df.rename(columns={
    "Model": "model",
    "Mean_Acc": "accuracy",
    "Mean_F1": "f1_score",
    "Std_Acc": "std_accuracy",
    "Std_F1": "std_f1",
    "SE_Acc": "se_accuracy",
    "SE_F1": "se_f1",
    "Mean_Time(s)": "time_elapsed"
}, inplace=True)

results_df = results_df[[
    "model", "preprocessing", "n_components", "alpha",
    "accuracy", "f1_score",
    "std_accuracy", "std_f1",
    "se_accuracy", "se_f1", "time_elapsed"
]]

results_df = results_df.round(3)

# Thyroid Dataset



In [ ]:
import time
import numpy as np
import pandas as pd
from scipy.stats import sem
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, f1_score
from contrastive import CPCA
from xgboost import XGBClassifier
from sklearn.svm import SVC
from tabpfn import TabPFNClassifier
from tabicl import TabICLClassifier

# ============ 0. SAVE RESULTS TO GOOGLE DRIVE ============
from google.colab import drive
drive.mount('/content/drive')
save_path = "/content/drive/MyDrive/thyroid_comparison_results.csv"

# ============ 1. LOAD DATASET ============
df = pd.read_csv("cleaned_dataset_Thyroid1.csv")

# ============ 2. CREATE LABELS ============
df["query hypothyroid"] = df["query hypothyroid"].astype(int)
df["query hyperthyroid"] = df["query hyperthyroid"].astype(int)

def assign_label(row):
    hypo, hyper = row["query hypothyroid"], row["query hyperthyroid"]
    if hypo == 0 and hyper == 0:
        return 0
    elif hypo == 1 and hyper == 0:
        return 1
    elif hypo == 0 and hyper == 1:
        return 2
    else:
        return -1

df["label"] = df.apply(assign_label, axis=1)
df = df[df["label"] != -1]

# ============ 3. PREPARE FEATURES ============
X = df.drop(columns=["query hypothyroid", "query hyperthyroid", "label"], errors="ignore")
X = X.select_dtypes(include=[np.number]).fillna(0)
y = df["label"].values

# ============ 4. TRAIN/TEST SPLIT ============
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
scaler = StandardScaler()
X_train_full = scaler.fit_transform(X_train_full)
X_test = scaler.transform(X_test)

# ============ 5. DEFINE MODELS ============
models = {
    "TabPFN": lambda: TabPFNClassifier(device="cpu", ignore_pretraining_limits=True),
    "TabICL": lambda: TabICLClassifier(),
    "XGBoost": lambda: XGBClassifier(
        n_estimators=300, max_depth=5, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, eval_metric="logloss", random_state=42
    ),
    "SVC": lambda: SVC(kernel="rbf", probability=False, random_state=42)
}

# ============ 6. SETTINGS ============
pca_components = [2, 10, 20]
cpca_alphas = [1, 10, 100]
n_splits = 5
random_state = 42

# ============ 7. CROSS-VALIDATION LOOP ============
results = []
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)

for model_name, model_fn in models.items():
    print(f"\n=== Evaluating {model_name} ===")

    feature_sets = {"raw": X_train_full}

    # --- PCA variants ---
    for n_comp in pca_components:
        pca = PCA(n_components=n_comp, random_state=42)
        feature_sets[f"pca_{n_comp}"] = pca.fit_transform(X_train_full)

    # --- cPCA variants: vary components *and* alpha ---
    bg_mask = y_train_full == 0
    fg_mask = ~bg_mask
    for n_comp in pca_components:
        cpca = CPCA(n_components=n_comp, standardize=True)
        cpca.fit(X_train_full[fg_mask], X_train_full[bg_mask])

        for alpha in cpca_alphas:
            transformed_list, _ = cpca.transform(X_train_full, alpha_value=alpha, return_alphas=True)
            X_cpca = transformed_list[0]
            feature_sets[f"cpca_{n_comp}_alpha{alpha}"] = X_cpca

    # --- Run CV for each feature set ---
    for feat_name, X_data in feature_sets.items():
        print(f"\n>> {model_name} on {feat_name}")
        accs, f1s, times = [], [], []

        for train_idx, val_idx in skf.split(X_data, y_train_full):
            X_train, X_val = X_data[train_idx], X_data[val_idx]
            y_train, y_val = y_train_full[train_idx], y_train_full[val_idx]

            model = model_fn()

            start = time.time()
            model.fit(X_train, y_train)
            y_pred = model.predict(X_val)
            elapsed = time.time() - start

            accs.append(accuracy_score(y_val, y_pred))
            f1s.append(f1_score(y_val, y_pred, average="weighted"))
            times.append(elapsed)

        results.append({
            "Model": model_name,
            "Features": feat_name,
            "Mean_Acc": np.mean(accs),
            "Std_Acc": np.std(accs),
            "SE_Acc": sem(accs),
            "Mean_F1": np.mean(f1s),
            "Std_F1": np.std(f1s),
            "SE_F1": sem(f1s),
            "Mean_Time(s)": np.mean(times)
        })

# ============ 8. REPORT RESULTS ============
results_df = pd.DataFrame(results)

# Split the 'Features' column into preprocessing, n_components, alpha
def parse_features(name):
    if name == "raw":
        return "None", 0, "N/A"
    elif name.startswith("pca_"):
        n = int(name.split("_")[1])
        return "PCA", n, "N/A"
    elif name.startswith("cpca_"):
        parts = name.split("_")
        n = int(parts[1])
        alpha = parts[2].replace("alpha", "")
        return "cPCA", n, alpha
    else:
        return "Unknown", "N/A", "N/A"

results_df[["preprocessing", "n_components", "alpha"]] = results_df["Features"].apply(
    lambda x: pd.Series(parse_features(x))
)

# Rename columns to match publication format
results_df.rename(columns={
    "Model": "model",
    "Mean_Acc": "accuracy",
    "Mean_F1": "f1_score",
    "Std_Acc": "std_accuracy",
    "Std_F1": "std_f1",
    "SE_Acc": "se_accuracy",
    "SE_F1": "se_f1",
    "Mean_Time(s)": "time_elapsed"
}, inplace=True)

# Reorder columns
results_df = results_df[[
    "model", "preprocessing", "n_components", "alpha",
    "accuracy", "f1_score",
    "std_accuracy", "std_f1",
    "se_accuracy", "se_f1", "time_elapsed"
]]

# Round to 3 decimals for clarity
results_df = results_df.round(3)

# ============ 9. SAVE TO GOOGLE DRIVE ============
pd.set_option("display.max_rows", None)
print("\n=== Final Table (Publication Format) ===")
print(results_df)

results_df.to_csv(save_path, index=False)
print(f"\n✅ Publication-style table saved to Google Drive at: {save_path}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

=== Evaluating TabPFN ===


/usr/local/lib/python3.12/dist-packages/contrastive/__init__.py:40: RuntimeWarning: invalid value encountered in divide
  standardized_array =  (array-np.mean(array,axis=0)) / np.std(array,axis=0)
/usr/local/lib/python3.12/dist-packages/contrastive/__init__.py:40: RuntimeWarning: invalid value encountered in divide
  standardized_array =  (array-np.mean(array,axis=0)) / np.std(array,axis=0)
/usr/local/lib/python3.12/dist-packages/contrastive/__init__.py:40: RuntimeWarning: invalid value encountered in divide
  standardized_array =  (array-np.mean(array,axis=0)) / np.std(array,axis=0)



>> TabPFN on raw


tabpfn-v2-classifier-finetuned-zk73skhh.(…):   0%|          | 0.00/29.0M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/37.0 [00:00<?, ?B/s]

...


>> TabICL on raw

>> TabICL on pca_2

>> TabICL on pca_10

>> TabICL on pca_20

>> TabICL on cpca_2_alpha1

>> TabICL on cpca_2_alpha10

>> TabICL on cpca_2_alpha100

>> TabICL on cpca_10_alpha1

>> TabICL on cpca_10_alpha10

>> TabICL on cpca_10_alpha100

>> TabICL on cpca_20_alpha1

>> TabICL on cpca_20_alpha10

>> TabICL on cpca_20_alpha100

=== Evaluating XGBoost ===


/usr/local/lib/python3.12/dist-packages/contrastive/__init__.py:40: RuntimeWarning: invalid value encountered in divide
  standardized_array =  (array-np.mean(array,axis=0)) / np.std(array,axis=0)
/usr/local/lib/python3.12/dist-packages/contrastive/__init__.py:40: RuntimeWarning: invalid value encountered in divide
  standardized_array =  (array-np.mean(array,axis=0)) / np.std(array,axis=0)
/usr/local/lib/python3.12/dist-packages/contrastive/__init__.py:40: RuntimeWarning: invalid value encountered in divide
  standardized_array =  (array-np.mean(array,axis=0)) / np.std(array,axis=0)



>> XGBoost on raw

>> XGBoost on pca_2

>> XGBoost on pca_10

>> XGBoost on pca_20

>> XGBoost on cpca_2_alpha1

>> XGBoost on cpca_2_alpha10

>> XGBoost on cpca_2_alpha100

>> XGBoost on cpca_10_alpha1

>> XGBoost on cpca_10_alpha10

>> XGBoost on cpca_10_alpha100

>> XGBoost on cpca_20_alpha1

>> XGBoost on cpca_20_alpha10

>> XGBoost on cpca_20_alpha100

=== Evaluating SVC ===


/usr/local/lib/python3.12/dist-packages/contrastive/__init__.py:40: RuntimeWarning: invalid value encountered in divide
  standardized_array =  (array-np.mean(array,axis=0)) / np.std(array,axis=0)
/usr/local/lib/python3.12/dist-packages/contrastive/__init__.py:40: RuntimeWarning: invalid value encountered in divide
  standardized_array =  (array-np.mean(array,axis=0)) / np.std(array,axis=0)
/usr/local/lib/python3.12/dist-packages/contrastive/__init__.py:40: RuntimeWarning: invalid value encountered in divide
  standardized_array =  (array-np.mean(array,axis=0)) / np.std(array,axis=0)



>> SVC on raw

>> SVC on pca_2

>> SVC on pca_10

>> SVC on pca_20

>> SVC on cpca_2_alpha1

>> SVC on cpca_2_alpha10

>> SVC on cpca_2_alpha100

>> SVC on cpca_10_alpha1

>> SVC on cpca_10_alpha10

>> SVC on cpca_10_alpha100

>> SVC on cpca_20_alpha1

>> SVC on cpca_20_alpha10

>> SVC on cpca_20_alpha100

=== Final Table (Publication Format) ===
      model preprocessing  n_components alpha  accuracy  f1_score  \
0    TabPFN          None             0   N/A     0.885     0.833   
1    TabPFN           PCA             2   N/A     0.885     0.832   
2    TabPFN           PCA            10   N/A     0.885     0.833   
3    TabPFN           PCA            20   N/A     0.885     0.833   
4    TabPFN          cPCA             2     1     0.885     0.832   
5    TabPFN          cPCA             2    10     0.885     0.832   
6    TabPFN          cPCA             2   100     0.885     0.832   
7    TabPFN          cPCA            10     1     0.884     0.832   
8    TabPFN          cPCA    